In [28]:
import pandas as pd, json, html as html_lib
from IPython.display import display, HTML

df = pd.read_csv('Dataset-Boston-2019.csv', sep=';', on_bad_lines='skip')
df['Result_sec'] = pd.to_numeric(df['Result_sec'], errors='coerce')
df = df.dropna(subset=['Result_sec', 'Country', 'Country_code'])

ps = df.groupby(['Country', 'Country_code']).agg(
    Total=('Result_sec', 'count'),
    Promedio=('Result_sec', 'mean')
).reset_index()
ps = ps[ps['Total'] >= 10].sort_values('Total', ascending=False).reset_index(drop=True)

# Mapeo completo IOC/ISO-3 a ISO-2 para generar la bandera exacta de cada país
iso3_to_2 = {
    'USA':'US','CAN':'CA','GBR':'GB','MEX':'MX','CHN':'CN','JPN':'JP','GER':'DE','BRA':'BR','AUS':'AU','FRA':'FR',
    'ITA':'IT','ESP':'ES','KOR':'KR','NED':'NL','SUI':'CH','IRL':'IE','SWE':'SE','IND':'IN','COL':'CO','ARG':'AR',
    'CHI':'CL','POL':'PL','HKG':'HK','TPE':'TW','DEN':'DK','NOR':'NO','BEL':'BE','AUT':'AT','RSA':'ZA','NZL':'NZ',
    'RUS':'RU','ISR':'IL','SGP':'SG','CRC':'CR','FIN':'FI','PER':'PE','CZE':'CZ','POR':'PT','PHI':'PH','VEN':'VE',
    'GUA':'GT','ECU':'EC','THA':'TH','MAS':'MY','UKR':'UA','TUR':'TR','HUN':'HU','ROU':'RO','GRE':'GR','ISL':'IS',
    'PAN':'PA','URU':'UY','DOM':'DO','PUR':'PR','ESA':'SV','HON':'HN','BOL':'BO','PAR':'PY','NCA':'NI','KEN':'KE',
    'ETH':'ET','MAR':'MA','EGY':'EG','TUN':'TN','NGR':'NG','UAE':'AE','KSA':'SA','QAT':'QA','KUW':'KW','LIB':'LB',
    'JOR':'JO','BAH':'BS','JAM':'JM','TRI':'TT','BAR':'BB','BER':'BM','CAY':'KY','LUX':'LU','SLO':'SI','SVK':'SK',
    'CRO':'HR','SRB':'RS','BUL':'BG','EST':'EE','LAT':'LV','LTU':'LT','CYP':'CY','MLT':'MT','IDN':'ID','VIE':'VN'
}

def get_flag(code):
    c2 = iso3_to_2.get(str(code).strip().upper(), '')
    return ''.join(chr(127397 + ord(ch)) for ch in c2) if len(c2) == 2 else '🏳️'

ps['Flag'] = ps['Country_code'].apply(get_flag)
ps['Vel'] = 42.195 / (ps['Promedio'] / 60)

datos_js = json.dumps(ps.to_dict(orient='records'))
def make_opts(code):
    return "".join([f"<option value='{i}' {'selected' if r['Country_code']==code else ''}>{r['Flag']} {r['Country']}</option>" for i, r in ps.iterrows()])

o0, o1, o2, o3 = make_opts('ARG'), make_opts('KEN'), make_opts('BRA'), make_opts('USA')
print("Listo:", len(ps), "países con banderas verificadas.")

Listo: 78 países con banderas verificadas.


In [30]:
app = f"""<!DOCTYPE html><html><body style="margin:0;background:transparent;font-family:sans-serif;color:#f8fafc;">
<style>
  @keyframes run {{ 0%,100%{{transform:translateY(0) scaleX(-1)}} 50%{{transform:translateY(-6px) scaleX(-1)}} }}
  @keyframes win {{ 0%,100%{{transform:scale(1)}} 50%{{transform:scale(1.3) rotate(10deg)}} }}
  .run {{ display:inline-block; animation:run .35s infinite; font-size:26px; }}
  .idle {{ display:inline-block; transform:scaleX(-1); font-size:26px; }}
  .win {{ display:inline-block; animation:win .5s infinite; font-size:30px; }}
</style>
<div style="background:linear-gradient(180deg,#0f172a,#1e293b);padding:18px;border-radius:16px;border:2px solid #38bdf8;">
  <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:12px;">
    <h3 style="margin:0;color:#38bdf8;">🏟️ Pista Olímpica Boston 2019</h3>
    <div id="clk" style="background:#020617;padding:5px 14px;border-radius:20px;font-weight:bold;color:#facc15;border:1px solid #facc15;">⏱️ 0:00 hs</div>
  </div>
  <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:8px;margin-bottom:12px;">
    <div><small style="color:#facc15;font-weight:bold;">⭐ Tu País:</small><select id="s0" onchange="rst()" style="width:100%;padding:6px;border-radius:6px;background:#020617;color:#fff;border:1px solid #facc15;">{o0}</select></div>
    <div><small style="color:#38bdf8;">Rival 1:</small><select id="s1" onchange="rst()" style="width:100%;padding:6px;border-radius:6px;background:#020617;color:#fff;border:1px solid #38bdf8;">{o1}</select></div>
    <div><small style="color:#38bdf8;">Rival 2:</small><select id="s2" onchange="rst()" style="width:100%;padding:6px;border-radius:6px;background:#020617;color:#fff;border:1px solid #38bdf8;">{o2}</select></div>
    <div><small style="color:#38bdf8;">Rival 3:</small><select id="s3" onchange="rst()" style="width:100%;padding:6px;border-radius:6px;background:#020617;color:#fff;border:1px solid #38bdf8;">{o3}</select></div>
  </div>
  <div style="display:flex;gap:8px;margin-bottom:12px;">
    <button onclick="go()" style="background:#22c55e;color:#052e16;border:0;padding:8px 16px;border-radius:8px;font-weight:bold;cursor:pointer;">▶ PLAY</button>
    <button onclick="stop()" style="background:#f59e0b;color:#451a03;border:0;padding:8px 14px;border-radius:8px;font-weight:bold;cursor:pointer;">⏸ PAUSA</button>
    <button onclick="rst()" style="background:#ef4444;color:#fff;border:0;padding:8px 14px;border-radius:8px;font-weight:bold;cursor:pointer;">↺ REINICIAR</button>
  </div>
  <div style="background:#7c2d12;padding:10px 12px;border-radius:12px;border:3px solid #fdba74;position:relative;overflow:hidden;">
    <div style="position:absolute;right:42px;top:0;bottom:0;width:6px;background:repeating-linear-gradient(0deg,#fff,#fff 8px,#000 8px,#000 16px);"></div>
    <div id="lanes"></div>
  </div>
  <div id="msg" style="margin-top:10px;text-align:center;font-weight:bold;color:#a7f3d0;"></div>
</div>
<script>
const P={datos_js}; let t=null, m=0, C=[];
function build(){{
  C=[0,1,2,3].map(i=>P[document.getElementById('s'+i).value]);
  document.getElementById('lanes').innerHTML=C.map((c,i)=>`<div style="border-bottom:1px dashed rgba(255,255,255,.3);padding:8px 0;">
    <div style="font-size:12px;display:flex;justify-content:space-between;"><span><b>Carril ${{i+1}}:</b> ${{c.Flag}} ${{c.Country}} ${{i===0?'⭐':''}}</span><span id="k${{i}}" style="margin-right:48px;">0.0 km</span></div>
    <div style="height:30px;position:relative;"><div id="r${{i}}" style="position:absolute;left:0%;transition:left .08s linear;white-space:nowrap;"><span id="ic${{i}}" class="idle">🏃‍♂️</span><span>${{c.Flag}}</span></div></div>
  </div>`).join('');
}}
function go(){{
  if(t)return; document.getElementById('msg').innerText="🔥 ¡Largaron!";
  C.forEach((_,i)=>document.getElementById('ic'+i).className='run');
  t=setInterval(()=>{{
    m+=1.5; document.getElementById('clk').innerText=`⏱️ ${{Math.floor(m/60)}}:${{String(Math.floor(m%60)).padStart(2,'0')}} hs`;
    let fin=0;
    C.forEach((c,i)=>{{
      let d=Math.min(42.195,m*c.Vel); document.getElementById('r'+i).style.left=(d/42.195*85)+'%';
      if(d>=42.195){{fin++; document.getElementById('k'+i).innerText="🏆 ¡META!"; document.getElementById('ic'+i).innerText="🙌"; document.getElementById('ic'+i).className="win";}}
      else document.getElementById('k'+i).innerText=d.toFixed(1)+" km";
    }});
    if(fin===4){{stop(); document.getElementById('msg').innerText="🎉✨ ¡CARRERA FINALIZADA! ✨🎉";}}
  }},70);
}}
function stop(){{clearInterval(t);t=null;C.forEach((_,i)=>{{if(document.getElementById('ic'+i).className=='run')document.getElementById('ic'+i).className='idle';}});}}
function rst(){{stop();m=0;document.getElementById('clk').innerText="⏱️ 0:00 hs";document.getElementById('msg').innerText="Elegí los 4 países y apretá ▶ PLAY";build();}}
rst();
</script></body></html>"""

display(HTML(f'<iframe srcdoc="{html_lib.escape(app)}" style="width:100%;height:460px;border:none;"></iframe>'))